#### FIFTH ATTEMPT

## Temporal Attack Prediction

### OpTC + Windows-APT datasets

**Overview:**

The combined dataset integrates OpTC endpoint telemetry with Windows-APT 2025 telemetry to provide a broader set of benign and attack-related system activities for temporal cyberattack prediction. The two datasets were harmonised into a common schema containing timestamp, event_action, event_object, protocol, process and thread identifiers (pid, ppid, tid), hostname, label, and dataset_source.
For the LSTM and GRU experiment, timestamps are used to chronologically organise events and construct temporal sequences, allowing the models to learn patterns in preceding system activity and predict whether malicious activity will occur within a subsequent time window.

**Project goal:**

The aim is temporal prediction rather than event-level detection: use activity from the previous 10 minutes to predict whether malicious activity will occur during the next 5 minutes. The raw timestamp is retained for chronological ordering and window construction. hour and minute are also retained as model features, consistent with the previous experiment. Windows are created separately within each dataset source, hostname and date so sequences do not cross hosts, datasets or day boundaries.



In [121]:
# Imports
%matplotlib inline

from pathlib import Path
import gc
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from xgboost import XGBClassifier


import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    LSTM,
    GRU,
    Dense,
    Dropout,
    BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

seed = 7

random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

In [122]:
#command used to mount drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Create Temporal Combination Dataset

In [123]:
# optc_path = Path(
#     "/content/drive/MyDrive/solutions/OpTC_sample2_cleaned"
# )

# files = list(
#     optc_path.glob("*.parquet")
# )

# print("Number of OpTC files:", len(files))

# selected_features = [
#     "action",
#     "object",
#     "l4protocol",
#     "pid",
#     "ppid",
#     "tid",
#     "timestamp",
#     "label",
#     "hostname"
# ]

# data_parts = []

# for file in files:

#     part = pd.read_parquet(
#         file,
#         columns=selected_features
#     )

#     data_parts.append(part)

# optc = pd.concat(
#     data_parts,
#     ignore_index=True
# )

# del data_parts
# del part
# gc.collect()

# print("OpTC shape:", optc.shape)

In [124]:
# optc.rename(
#     columns={
#         "action": "event_action",
#         "object": "event_object",
#         "l4protocol": "protocol"
#     },
#     inplace=True
# )

# optc["dataset_source"] = "OpTC"


# # Standardise hostname first
# optc["hostname"] = (
#     optc["hostname"]
#     .astype("string")
#     .str.strip()
#     .str.lower()
# )


# # Standardise numeric fields
# for col in [
#     "pid",
#     "ppid",
#     "tid"
# ]:

#     optc[col] = (
#         pd.to_numeric(
#             optc[col],
#             errors="coerce"
#         )
#         .fillna(-1)
#         .astype("float32")
#     )

#     gc.collect()


# # Keep timestamp as string for source-specific
# # conversion later
# optc["timestamp"] = (
#     optc["timestamp"]
#     .astype("string")
#     .fillna("")
# )


# # Standardise categorical fields
# for col in [
#     "event_action",
#     "event_object",
#     "protocol"
# ]:

#     optc[col] = (
#         optc[col]
#         .astype("string")
#         .fillna("UNKNOWN")
#     )


# optc["label"] = (
#     pd.to_numeric(
#         optc["label"],
#         errors="coerce"
#     )
#     .fillna(0)
#     .astype("int8")
# )


# harmonised_features = [
#     "timestamp",
#     "hostname",
#     "event_action",
#     "event_object",
#     "pid",
#     "ppid",
#     "tid",
#     "protocol",
#     "label",
#     "dataset_source"
# ]

# optc = optc[
#     harmonised_features
# ]

# gc.collect()

# print("OpTC standardised:", optc.shape)
# print(optc.dtypes)

In [125]:
# optc_test_hosts = [
#     "sysclient0351.systemia.com",
#     "sysclient0352.systemia.com"
# ]

# optc_test_mask = (
#     optc["hostname"]
#     .isin(optc_test_hosts)
# )

# print("OpTC train rows:", (~optc_test_mask).sum())
# print("OpTC test rows:", optc_test_mask.sum())

# print("\nHeld-out hosts:")

# print(
#     optc.loc[
#         optc_test_mask,
#         "hostname"
#     ].value_counts()
# )

# assert optc_test_mask.sum() > 0, (
#     "No OpTC test rows were found."
# )

In [126]:
# apt_path = Path(
#     "/content/drive/MyDrive/solutions/"
#     "Windows_APT/combined.csv"
# )

# mapping_path = Path(
#     "/content/drive/MyDrive/solutions/"
#     "Windows_APT/log_to_scenario_mapping.csv"
# )

# apt = pd.read_csv(
#     apt_path,
#     low_memory=False
# )

# mapping = pd.read_csv(
#     mapping_path,
#     low_memory=False
# )

# print("Windows-APT:", apt.shape)
# print("Mapping:", mapping.shape)

In [127]:
# # add scenario mapping and proxy labels
# mitre_id_clean = (
#     apt["_source.rule.mitre.id"]
#     .fillna("")
#     .astype(str)
#     .str.strip()
# )

# has_mitre = ~mitre_id_clean.isin(
#     [
#         "",
#         "[]",
#         "nan",
#         "None"
#     ]
# )


# apt_mapped = pd.concat(
#     [
#         apt.reset_index(
#             drop=True
#         ),

#         mapping[
#             [
#                 "Scenario_ID",
#                 "Scenario_Name",
#                 "MITRE_Group_ID",
#                 "Scenario_Posterior",
#                 "Derived_Label",
#                 "Attribution_Strength",
#                 "Mapping_Evidence",
#                 "Candidate_Scenarios"
#             ]
#         ].reset_index(
#             drop=True
#         )
#     ],
#     axis=1
# )


# apt_mapped["label"] = (
#     has_mitre
#     .astype("int8")
# )

# apt_mapped["dataset_source"] = (
#     "Windows-APT"
# )

# del mitre_id_clean
# del has_mitre
# del mapping
# del apt

# gc.collect()

# print(
#     apt_mapped["label"]
#     .value_counts()
# )

In [128]:
# apt_mapped.rename(
#     columns={
#         "_source.@timestamp":
#             "timestamp",

#         "_source.data.win.system.computer":
#             "hostname",

#         "_source.rule.description":
#             "rule_description",

#         "_source.data.win.eventdata.eventType":
#             "event_type",

#         "_source.data.win.eventdata.type":
#             "type",

#         "_source.data.win.eventdata.processId":
#             "pid",

#         "_source.data.win.eventdata.parentProcessId":
#             "ppid",

#         "_source.data.win.system.threadID":
#             "tid",

#         "_source.data.win.eventdata.protocol":
#             "protocol",

#         "Scenario_ID":
#             "scenario_id",

#         "Scenario_Name":
#             "scenario_name"
#     },
#     inplace=True
# )


# apt_mapped["event_action"] = (
#     apt_mapped["rule_description"]
#     .fillna(
#         apt_mapped["event_type"]
#     )
# )

# apt_mapped["event_object"] = (
#     apt_mapped["event_type"]
#     .fillna(
#         apt_mapped["type"]
#     )
# )

In [129]:
# apt_mapped["hostname"] = (
#     apt_mapped["hostname"]
#     .astype("string")
#     .str.strip()
#     .str.lower()
# )


# for col in [
#     "pid",
#     "ppid",
#     "tid"
# ]:

#     apt_mapped[col] = (
#         pd.to_numeric(
#             apt_mapped[col],
#             errors="coerce"
#         )
#         .fillna(-1)
#         .astype("float32")
#     )


# apt_mapped["timestamp"] = (
#     apt_mapped["timestamp"]
#     .astype("string")
#     .fillna("")
# )


# for col in [
#     "event_action",
#     "event_object",
#     "protocol"
# ]:

#     apt_mapped[col] = (
#         apt_mapped[col]
#         .astype("string")
#         .fillna("UNKNOWN")
#     )


# apt_mapped["label"] = (
#     apt_mapped["label"]
#     .astype("int8")
# )


# print(
#     apt_mapped[
#         [
#             "timestamp",
#             "hostname",
#             "event_action",
#             "event_object",
#             "pid",
#             "ppid",
#             "tid",
#             "protocol",
#             "label",
#             "dataset_source"
#         ]
#     ].dtypes
# )

In [130]:
# apt_groups = (
#     apt_mapped["scenario_id"]
#     .fillna("UNRESOLVED")
#     .astype(str)
# )


# splitter = GroupShuffleSplit(
#     n_splits=1,
#     test_size=0.20,
#     random_state=7
# )


# apt_train_idx, apt_test_idx = next(
#     splitter.split(
#         apt_mapped,
#         apt_mapped["label"],
#         groups=apt_groups
#     )
# )


# print(
#     "APT train rows:",
#     len(apt_train_idx)
# )

# print(
#     "APT test rows:",
#     len(apt_test_idx)
# )

In [131]:
# harmonised_features = [
#     "timestamp",
#     "hostname",
#     "event_action",
#     "event_object",
#     "pid",
#     "ppid",
#     "tid",
#     "protocol",
#     "label",
#     "dataset_source"
# ]


# apt_train = (
#     apt_mapped
#     .iloc[apt_train_idx][harmonised_features]
#     .copy()
# )


# apt_test = (
#     apt_mapped
#     .iloc[apt_test_idx][harmonised_features]
#     .copy()
# )


# print(
#     "APT train:",
#     apt_train.shape
# )

# print(
#     "APT test:",
#     apt_test.shape
# )


# del apt_groups
# del apt_train_idx
# del apt_test_idx
# del apt_mapped

# gc.collect()

In [132]:
# temporal_combination_path = Path(
#     "/content/drive/MyDrive/solutions/"
#     "Temporal_Combination_OpTC_APT"
# )

# temporal_combination_path.mkdir(
#     parents=True,
#     exist_ok=True
# )

In [133]:
# optc_train_part = optc.loc[
#     ~optc_test_mask
# ]


# print(
#     "OpTC train:",
#     optc_train_part.shape
# )

# print(
#     "APT train:",
#     apt_train.shape
# )


# temporal_train = pd.concat(
#     [
#         optc_train_part,
#         apt_train
#     ],
#     ignore_index=True
# )


# print(
#     "Combined train:",
#     temporal_train.shape
# )

# print(
#     temporal_train[
#         "dataset_source"
#     ].value_counts()
# )


# temporal_train.to_parquet(
#     temporal_combination_path /
#     "temporal_train.parquet",
#     index=False
# )


# print(
#     "temporal_train.parquet saved."
# )


# del temporal_train
# del optc_train_part
# del apt_train

# gc.collect()

In [134]:
# optc_test_part = optc.loc[
#     optc_test_mask
# ]


# print(
#     "OpTC test:",
#     optc_test_part.shape
# )

# print(
#     "APT test:",
#     apt_test.shape
# )


# temporal_test = pd.concat(
#     [
#         optc_test_part,
#         apt_test
#     ],
#     ignore_index=True
# )


# print(
#     "Combined test:",
#     temporal_test.shape
# )

# print(
#     temporal_test[
#         "dataset_source"
#     ].value_counts()
# )

# print("\nOpTC test hosts:")

# print(
#     temporal_test.loc[
#         temporal_test[
#             "dataset_source"
#         ].eq("OpTC"),
#         "hostname"
#     ].value_counts()
# )


# temporal_test.to_parquet(
#     temporal_combination_path /
#     "temporal_test.parquet",
#     index=False
# )


# print(
#     "temporal_test.parquet saved."
# )


# del temporal_test
# del optc_test_part
# del apt_test

# gc.collect()

In [135]:
# del optc
# del optc_test_mask

# gc.collect()

# print("Dataset creation memory cleared.")

### Load Combined Dataset

In [136]:
combination_path = Path(
    "/content/drive/MyDrive/solutions/Temporal_Combination_OpTC_APT"
)

train_data = pd.read_parquet(
    combination_path / "temporal_train.parquet"
)

test_data = pd.read_parquet(
    combination_path / "temporal_test.parquet"
)

print("Train shape:", train_data.shape)
print("Test shape:", test_data.shape)



Train shape: (11626747, 10)
Test shape: (5031118, 10)


### Prepare the timestamp and feature datatypes (Same as previous experiment)

In [137]:
def convert_timestamp_by_source(df):

    result = pd.Series(
        pd.NaT,
        index=df.index,
        dtype="datetime64[ns, UTC]"
    )

    # Windows-APT
    apt_mask = df["dataset_source"].eq("Windows-APT")

    result.loc[apt_mask] = pd.to_datetime(
        df.loc[apt_mask, "timestamp"],
        format="%b %d, %Y @ %H:%M:%S.%f",
        errors="coerce",
        utc=True
    )

    # OpTC
    optc_mask = df["dataset_source"].eq("OpTC")

    optc_raw = df.loc[
        optc_mask,
        "timestamp"
    ]

    numeric = pd.to_numeric(
        optc_raw,
        errors="coerce"
    )

    numeric_mask = numeric.notna()

    # Numeric OpTC timestamps
    if numeric_mask.any():

        values = numeric[numeric_mask]

        median_value = (
            values.abs().median()
        )

        if median_value > 1e17:
            unit = "ns"
        elif median_value > 1e14:
            unit = "us"
        elif median_value > 1e11:
            unit = "ms"
        else:
            unit = "s"

        result.loc[
            values.index
        ] = pd.to_datetime(
            values,
            unit=unit,
            errors="coerce",
            utc=True
        )

    # String-formatted OpTC timestamps
    string_idx = optc_raw.index[
        ~numeric_mask
    ]

    result.loc[
        string_idx
    ] = pd.to_datetime(
        optc_raw.loc[string_idx],
        errors="coerce",
        utc=True
    )

    return result


train_data["timestamp"] = (
    convert_timestamp_by_source(
        train_data
    )
)

test_data["timestamp"] = (
    convert_timestamp_by_source(
        test_data
    )
)


print("TRAIN")
print(
    train_data.groupby(
        "dataset_source"
    )["timestamp"].apply(
        lambda x: x.notna().sum()
    )
)

print("\nTEST")
print(
    test_data.groupby(
        "dataset_source"
    )["timestamp"].apply(
        lambda x: x.notna().sum()
    )
)

TRAIN
dataset_source
OpTC           11550333
Windows-APT       65461
Name: timestamp, dtype: int64

TEST
dataset_source
OpTC           4989495
Windows-APT      36550
Name: timestamp, dtype: int64


In [138]:
# Ensure numeric fields are numeric
for col in ["pid", "ppid", "tid"]:

    train_data[col] = pd.to_numeric(
        train_data[col],
        errors="coerce"
    ).fillna(-1).astype("float32")

    test_data[col] = pd.to_numeric(
        test_data[col],
        errors="coerce"
    ).fillna(-1).astype("float32")


# Standardise categorical fields
for col in [
    "event_action",
    "event_object",
    "protocol"
]:

    train_data[col] = (
        train_data[col]
        .fillna("UNKNOWN")
        .astype(str)
    )

    test_data[col] = (
        test_data[col]
        .fillna("UNKNOWN")
        .astype(str)
    )



### Feature Engineering

In [139]:
# I aggregate raw events into one-minute intervals/chunks
# For each chunk, the model receives total event count, unique process/thread counts,
# hour and minute, counts of the most common event actions, objects and protocols.

# Temporal settings
TIME_BIN = "1min"
LOOKBACK_MINUTES = 10
PREDICT_AHEAD_MINUTES = 3

TOP_ACTIONS = 30
TOP_OBJECTS = 20
TOP_PROTOCOLS = 10


# Learn categorical vocabulary from training data only
top_actions = (
    train_data["event_action"]
    .value_counts()
    .head(TOP_ACTIONS)
    .index
    .tolist()
)

top_objects = (
    train_data["event_object"]
    .value_counts()
    .head(TOP_OBJECTS)
    .index
    .tolist()
)

top_protocols = (
    train_data["protocol"]
    .value_counts()
    .head(TOP_PROTOCOLS)
    .index
    .tolist()
)


def clean_feature_name(value):
    return (
        str(value)
        .strip()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(".", "_")
    )


action_columns = {
    value: f"action__{clean_feature_name(value)}"
    for value in top_actions
}

object_columns = {
    value: f"object__{clean_feature_name(value)}"
    for value in top_objects
}

protocol_columns = {
    value: f"protocol__{clean_feature_name(value)}"
    for value in top_protocols
}

In [140]:
def build_minute_features(data):

    # Use only the columns needed for aggregation
    df = data[
        [
            "dataset_source",
            "hostname",
            "timestamp",
            "pid",
            "ppid",
            "tid",
            "label",
            "event_action",
            "event_object",
            "protocol"
        ]
    ]

    # Create time bin only
    time_bin = (
        df["timestamp"]
        .dt.floor(TIME_BIN)
    )


    # Base minute-level features


    base = pd.DataFrame({
        "dataset_source": df["dataset_source"],
        "hostname": df["hostname"],
        "time_bin": time_bin,
        "pid": df["pid"],
        "ppid": df["ppid"],
        "tid": df["tid"],
        "label": df["label"]
    })

    minute_data = (
        base.groupby(
            [
                "dataset_source",
                "hostname",
                "time_bin"
            ],
            sort=False,
            observed=True
        )
        .agg(
            event_count=("label", "size"),
            unique_pid=("pid", "nunique"),
            unique_ppid=("ppid", "nunique"),
            unique_tid=("tid", "nunique"),
            attack_now=("label", "max")
        )
        .reset_index()
    )

    del base
    gc.collect()



    # Action counts


    action_counts = (
        pd.DataFrame({
            "dataset_source": df["dataset_source"],
            "hostname": df["hostname"],
            "time_bin": time_bin,
            "event_action": df["event_action"]
        })
        .groupby(
            [
                "dataset_source",
                "hostname",
                "time_bin",
                "event_action"
            ],
            sort=False,
            observed=True
        )
        .size()
        .reset_index(name="count")
    )

    # Filter AFTER aggregation
    action_counts = action_counts[
        action_counts["event_action"].isin(
            top_actions
        )
    ]

    action_counts = (
        action_counts.pivot_table(
            index=[
                "dataset_source",
                "hostname",
                "time_bin"
            ],
            columns="event_action",
            values="count",
            fill_value=0
        )
        .rename(columns=action_columns)
        .reset_index()
    )

    minute_data = minute_data.merge(
        action_counts,
        on=[
            "dataset_source",
            "hostname",
            "time_bin"
        ],
        how="left"
    )

    del action_counts
    gc.collect()



    # Object counts


    object_counts = (
        pd.DataFrame({
            "dataset_source": df["dataset_source"],
            "hostname": df["hostname"],
            "time_bin": time_bin,
            "event_object": df["event_object"]
        })
        .groupby(
            [
                "dataset_source",
                "hostname",
                "time_bin",
                "event_object"
            ],
            sort=False,
            observed=True
        )
        .size()
        .reset_index(name="count")
    )

    object_counts = object_counts[
        object_counts["event_object"].isin(
            top_objects
        )
    ]

    object_counts = (
        object_counts.pivot_table(
            index=[
                "dataset_source",
                "hostname",
                "time_bin"
            ],
            columns="event_object",
            values="count",
            fill_value=0
        )
        .rename(columns=object_columns)
        .reset_index()
    )

    minute_data = minute_data.merge(
        object_counts,
        on=[
            "dataset_source",
            "hostname",
            "time_bin"
        ],
        how="left"
    )

    del object_counts
    gc.collect()


    # Protocol counts


    protocol_counts = (
        pd.DataFrame({
            "dataset_source": df["dataset_source"],
            "hostname": df["hostname"],
            "time_bin": time_bin,
            "protocol": df["protocol"]
        })
        .groupby(
            [
                "dataset_source",
                "hostname",
                "time_bin",
                "protocol"
            ],
            sort=False,
            observed=True
        )
        .size()
        .reset_index(name="count")
    )

    protocol_counts = protocol_counts[
        protocol_counts["protocol"].isin(
            top_protocols
        )
    ]

    protocol_counts = (
        protocol_counts.pivot_table(
            index=[
                "dataset_source",
                "hostname",
                "time_bin"
            ],
            columns="protocol",
            values="count",
            fill_value=0
        )
        .rename(columns=protocol_columns)
        .reset_index()
    )

    minute_data = minute_data.merge(
        protocol_counts,
        on=[
            "dataset_source",
            "hostname",
            "time_bin"
        ],
        how="left"
    )

    del protocol_counts
    gc.collect()


    # Final temporal information


    minute_data = minute_data.fillna(0)

    minute_data["date"] = (
        minute_data["time_bin"]
        .dt.date
    )

    minute_data["hour"] = (
        minute_data["time_bin"]
        .dt.hour
        .astype("float32")
    )

    minute_data["minute"] = (
        minute_data["time_bin"]
        .dt.minute
        .astype("float32")
    )

    return minute_data

In [141]:
 train_minutes = build_minute_features(
    train_data
)

print(
    "Train minute aggregation complete:",
    train_minutes.shape
)

gc.collect()

test_minutes = build_minute_features(
    test_data
)

print(
    "Test minute aggregation complete:",
    test_minutes.shape
)

gc.collect()

Train minute aggregation complete: (18520, 62)
Test minute aggregation complete: (8684, 58)


21

### Create Target Feature

In [142]:
def add_future_target(data):

    output = []

    group_columns = [
        "dataset_source",
        "hostname",
        "date"
    ]

    for _, group in data.groupby(
        group_columns,
        sort=False
    ):

        group = group.sort_values(
            "time_bin"
        ).copy()

        future_labels = []

        for step in range(
            1,
            PREDICT_AHEAD_MINUTES + 1
        ):

            future_labels.append(
                group["attack_now"]
                .shift(-step)
                .fillna(0)
                .to_numpy()
            )

        group["future_attack"] = (
            np.max(
                np.vstack(future_labels),
                axis=0
            ) > 0
        ).astype("int8")

        output.append(group)

    result = pd.concat(
        output,
        ignore_index=True
    )

    # Exclude minutes where the attack has already started
    result = result[
        result["attack_now"] == 0
    ].reset_index(drop=True)

    return result


train_minutes = add_future_target(
    train_minutes
)

test_minutes = add_future_target(
    test_minutes
)

print("Training target distribution:")
print(
    train_minutes["future_attack"]
    .value_counts()
)

print("\nTest target distribution:")
print(
    test_minutes["future_attack"]
    .value_counts()
)

Training target distribution:
future_attack
0    5829
1    2046
Name: count, dtype: int64

Test target distribution:
future_attack
0    2030
1     494
Name: count, dtype: int64


In [143]:
## Clean DData for duplicates

train_minutes = build_minute_features(
    train_data
)

test_minutes = build_minute_features(
    test_data
)

# Remove duplicate columns after feature engineering
print("Train duplicates:")
print(
    train_minutes.columns[
        train_minutes.columns.duplicated()
    ].tolist()
)

print("\nTest duplicates:")
print(
    test_minutes.columns[
        test_minutes.columns.duplicated()
    ].tolist()
)

train_minutes = train_minutes.loc[
    :,
    ~train_minutes.columns.duplicated()
].copy()

test_minutes = test_minutes.loc[
    :,
    ~test_minutes.columns.duplicated()
].copy()

print(
    "\nTrain columns:",
    len(train_minutes.columns)
)

print(
    "Test columns:",
    len(test_minutes.columns)
)
train_minutes = add_future_target(train_minutes)
test_minutes = add_future_target(test_minutes)

Train duplicates:
[]

Test duplicates:
[]

Train columns: 62
Test columns: 58


### Scale and Encode

In [144]:
# Columns that should NOT be model features
metadata_columns = [
    "dataset_source",
    "hostname",
    "date",
    "time_bin",
    "attack_now",
    "future_attack"
]

# Define features from training data only
feature_columns = [
    col
    for col in train_minutes.columns
    if col not in metadata_columns
]

# Add any missing training columns to test
for col in feature_columns:
    if col not in test_minutes.columns:
        test_minutes[col] = 0

# Keep only the exact same feature columns
X_train_minutes = train_minutes[
    feature_columns
].copy()

X_test_minutes = test_minutes[
    feature_columns
].copy()

# Make everything numeric
X_train_minutes = X_train_minutes.apply(
    pd.to_numeric,
    errors="coerce"
).fillna(0).astype("float32")

X_test_minutes = X_test_minutes.apply(
    pd.to_numeric,
    errors="coerce"
).fillna(0).astype("float32")

print("Number of temporal features:", len(feature_columns))
print("Train feature shape:", X_train_minutes.shape)
print("Test feature shape:", X_test_minutes.shape)

assert X_train_minutes.shape[1] == len(feature_columns)
assert X_test_minutes.shape[1] == len(feature_columns)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train_minutes
).astype("float32")

X_test_scaled = scaler.transform(
    X_test_minutes
).astype("float32")

print("Scaled train shape:", X_train_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)

Number of temporal features: 57
Train feature shape: (7875, 57)
Test feature shape: (2524, 57)
Scaled train shape: (7875, 57)
Scaled test shape: (2524, 57)


### Create Temporal Sequences

In [145]:
def create_sequences(
    data,
    scaled_features,
    lookback=LOOKBACK_MINUTES
):

    X_sequences = []
    y_sequences = []
    metadata = []

    group_columns = [
        "dataset_source",
        "hostname",
        "date"
    ]

    # Keep row positions aligned with the scaled matrix
    working = data.reset_index(
        drop=True
    ).copy()

    working["_row_position"] = np.arange(
        len(working)
    )

    for _, group in working.groupby(
        group_columns,
        sort=False
    ):

        group = group.sort_values(
            "time_bin"
        )

        positions = group[
            "_row_position"
        ].to_numpy()

        labels = group[
            "future_attack"
        ].to_numpy()

        times = group[
            "time_bin"
        ].to_numpy()

        sources = group[
            "dataset_source"
        ].to_numpy()

        hosts = group[
            "hostname"
        ].to_numpy()

        for end_idx in range(
            lookback - 1,
            len(group)
        ):

            start_idx = (
                end_idx - lookback + 1
            )

            window_positions = positions[
                start_idx:end_idx + 1
            ]

            X_sequences.append(
                scaled_features[
                    window_positions
                ]
            )

            y_sequences.append(
                labels[end_idx]
            )

            metadata.append({
                "dataset_source":
                    sources[end_idx],
                "hostname":
                    hosts[end_idx],
                "window_end":
                    times[end_idx]
            })

    return (
        np.asarray(
            X_sequences,
            dtype=np.float32
        ),
        np.asarray(
            y_sequences,
            dtype=np.int8
        ),
        pd.DataFrame(metadata)
    )


X_train_seq, Y_train_seq, train_seq_meta = (
    create_sequences(
        train_minutes,
        X_train_scaled
    )
)

X_test_seq, Y_test_seq, test_seq_meta = (
    create_sequences(
        test_minutes,
        X_test_scaled
    )
)


print(
    "LSTM/GRU train shape:",
    X_train_seq.shape
)

print(
    "LSTM/GRU test shape:",
    X_test_seq.shape
)

print("\nTraining sequence labels:")
print(
    pd.Series(Y_train_seq)
    .value_counts()
)

LSTM/GRU train shape: (7084, 10, 57)
LSTM/GRU test shape: (2125, 10, 57)

Training sequence labels:
0    5454
1    1630
Name: count, dtype: int64


### Compute Class Weights

In [146]:
# Create internal train/validation split
# Final test set remains completely untouched

split_index = int(
    len(X_train_seq) * 0.80
)

X_fit = X_train_seq[
    :split_index
]

Y_fit = Y_train_seq[
    :split_index
]

X_internal_val = X_train_seq[
    split_index:
]

Y_internal_val = Y_train_seq[
    split_index:
]

print(
    "Model training:",
    X_fit.shape
)

print(
    "Internal validation:",
    X_internal_val.shape
)

print(
    "Final test:",
    X_test_seq.shape
)

print("\nTraining labels:")
print(
    pd.Series(Y_fit).value_counts()
)

print("\nInternal validation labels:")
print(
    pd.Series(Y_internal_val).value_counts()
)

Model training: (5667, 10, 57)
Internal validation: (1417, 10, 57)
Final test: (2125, 10, 57)

Training labels:
0    4851
1     816
Name: count, dtype: int64

Internal validation labels:
1    814
0    603
Name: count, dtype: int64


In [147]:
# classes = np.unique(Y_train_seq)

# weights = compute_class_weight(
#     class_weight="balanced",
#     classes=classes,
#     y=Y_train_seq
# )

# # class_weights = {
# #     int(cls): float(weight)
# #     for cls, weight in zip(classes, weights)
# # }

# class_weights = {
#     0: np.sqrt(0.6161996870999857),
#     1: np.sqrt(2.651468788249694)
# }


# print("Class weights:")
# print(class_weights)

from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(
    Y_fit
)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=Y_fit
)

balanced_weights = {
    int(cls): float(weight)
    for cls, weight
    in zip(
        classes,
        weights
    )
}


# Weaken the balanced weights
# as in your successful experiment

class_weights = {
    cls: np.sqrt(weight)
    for cls, weight
    in balanced_weights.items()
}


print(
    "Balanced:",
    balanced_weights
)

print(
    "Used:",
    class_weights
)

Balanced: {0: 0.5841063698206556, 1: 3.4724264705882355}
Used: {0: np.float64(0.7642685194489274), 1: np.float64(1.8634447860315677)}


### Train ML Models

In [148]:
# Flatten temporal sequences for classical ML models
X_train_temporal = X_train_seq.reshape(
    X_train_seq.shape[0],
    -1
)


# Logistic Regression
lgr_model = LogisticRegression(
    max_iter=1000,
    random_state=7,
    class_weight="balanced"
)

lgr_model.fit(
    X_train_temporal,
    Y_train_seq
)

# lgr_eval_pred = lgr_model.predict(
#     X_train_temporal
# )

# lgr_eval_pred = lgr_model.predict_proba(
#     X_train_temporal
# )[:, 1]

lgr_eval_prob = lgr_model.predict_proba(
    X_train_temporal
)[:, 1]

LGR_THRESHOLD = 0.5

lgr_eval_pred = (
    lgr_eval_prob >= LGR_THRESHOLD
).astype(int)



print("===== Logistic Regression - Evaluation =====")

print(
    "Accuracy:",
    metrics.accuracy_score(
        Y_train_seq,
        lgr_eval_pred
    )
)

print(
    "Confusion Matrix:\n",
    metrics.confusion_matrix(
        Y_train_seq,
        lgr_eval_pred
    )
)

print(
    "Classification Report:\n",
    metrics.classification_report(
        Y_train_seq,
        lgr_eval_pred,
        digits=4,
        zero_division=0
    )
)


# XGBoost
xgb_model = XGBClassifier(
    random_state=7,
    n_estimators=300,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="logloss",
    n_jobs=-1
)

xgb_model.fit(
    X_train_temporal,
    Y_train_seq
)

# xgb_eval_pred = xgb_model.predict(
#     X_train_temporal
# )

# xgb_eval_pred = xgb_model.predict_proba(
#     X_train_temporal
# )[:, 1]

xgb_eval_prob = xgb_model.predict_proba(
    X_train_temporal
)[:, 1]

XGB_THRESHOLD = 0.5

xgb_eval_pred = (
    xgb_eval_prob >= XGB_THRESHOLD
).astype(int)

print("\n===== XGBoost - Evaluation =====")

print(
    "Accuracy:",
    metrics.accuracy_score(
        Y_train_seq,
        xgb_eval_pred
    )
)

print(
    "Confusion Matrix:\n",
    metrics.confusion_matrix(
        Y_train_seq,
        xgb_eval_pred
    )
)

print(
    "Classification Report:\n",
    metrics.classification_report(
        Y_train_seq,
        xgb_eval_pred,
        digits=4,
        zero_division=0
    )
)

===== Logistic Regression - Evaluation =====
Accuracy: 0.7514116318464145
Confusion Matrix:
 [[3779 1675]
 [  86 1544]]
Classification Report:
               precision    recall  f1-score   support

           0     0.9777    0.6929    0.8110      5454
           1     0.4797    0.9472    0.6368      1630

    accuracy                         0.7514      7084
   macro avg     0.7287    0.8201    0.7239      7084
weighted avg     0.8631    0.7514    0.7709      7084


===== XGBoost - Evaluation =====
Accuracy: 0.8931394692264257
Confusion Matrix:
 [[5131  323]
 [ 434 1196]]
Classification Report:
               precision    recall  f1-score   support

           0     0.9220    0.9408    0.9313      5454
           1     0.7874    0.7337    0.7596      1630

    accuracy                         0.8931      7084
   macro avg     0.8547    0.8373    0.8455      7084
weighted avg     0.8910    0.8931    0.8918      7084



### ML Validation

In [149]:
# Flatten test temporal sequences for classical ML models
X_test_temporal = X_test_seq.reshape(
    X_test_seq.shape[0],
    -1
)


# # Logistic Regression Validation
# lgr_val_pred = lgr_model.predict(
#     X_test_temporal
# )

# lgr_eval_pred = lgr_model.predict_proba(
#     X_train_temporal
# )[:, 1]

lgr_val_prob = lgr_model.predict_proba(
    X_test_temporal
)[:, 1]

LGR_THRESHOLD = 0.5

lgr_val_pred = (
    lgr_val_prob >= LGR_THRESHOLD
).astype(int)

print("===== Logistic Regression - Validation =====")

print(
    "Accuracy:",
    metrics.accuracy_score(
        Y_test_seq,
        lgr_val_pred
    )
)

print(
    "Confusion Matrix:\n",
    metrics.confusion_matrix(
        Y_test_seq,
        lgr_val_pred
    )
)

print(
    "Classification Report:\n",
    metrics.classification_report(
        Y_test_seq,
        lgr_val_pred,
        digits=4,
        zero_division=0
    )
)


# # XGBoost Validation
# xgb_val_pred = xgb_model.predict(
#     X_test_temporal
# )

# lgr_eval_pred = lgr_model.predict_proba(
#     X_train_temporal
# )[:, 1]

xgb_val_prob = xgb_model.predict_proba(
    X_test_temporal
)[:, 1]

XGB_THRESHOLD = 0.5

xgb_val_pred = (
    xgb_val_prob >= XGB_THRESHOLD
).astype(int)

print("\n===== XGBoost - Validation =====")

print(
    "Accuracy:",
    metrics.accuracy_score(
        Y_test_seq,
        xgb_val_pred
    )
)

print(
    "Confusion Matrix:\n",
    metrics.confusion_matrix(
        Y_test_seq,
        xgb_val_pred
    )
)

print(
    "Classification Report:\n",
    metrics.classification_report(
        Y_test_seq,
        xgb_val_pred,
        digits=4,
        zero_division=0
    )
)

===== Logistic Regression - Validation =====
Accuracy: 0.912
Confusion Matrix:
 [[1711  165]
 [  22  227]]
Classification Report:
               precision    recall  f1-score   support

           0     0.9873    0.9120    0.9482      1876
           1     0.5791    0.9116    0.7083       249

    accuracy                         0.9120      2125
   macro avg     0.7832    0.9118    0.8282      2125
weighted avg     0.9395    0.9120    0.9201      2125


===== XGBoost - Validation =====
Accuracy: 0.9096470588235294
Confusion Matrix:
 [[1819   57]
 [ 135  114]]
Classification Report:
               precision    recall  f1-score   support

           0     0.9309    0.9696    0.9499      1876
           1     0.6667    0.4578    0.5429       249

    accuracy                         0.9096      2125
   macro avg     0.7988    0.7137    0.7464      2125
weighted avg     0.8999    0.9096    0.9022      2125



### Train and Evaluate LSTM

In [150]:
lstm_model = Sequential([
    Input(
        shape=(
            X_train_seq.shape[1],
            X_train_seq.shape[2]
        )
    ),

    LSTM(
        128,
        return_sequences=True
    ),

    BatchNormalization(),

    Dropout(
        0.25
    ),

    LSTM(
        64
    ),

    BatchNormalization(),

    Dropout(
        0.25
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(
        0.20
    ),

    Dense(
        32,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])


lstm_model.compile(
    optimizer=Adam(
        learning_rate=0.0005
    ),

    loss="binary_crossentropy",

    metrics=[
        "accuracy",

        tf.keras.metrics.Precision(
            name="precision"
        ),

        tf.keras.metrics.Recall(
            name="recall"
        ),

        tf.keras.metrics.AUC(
            name="auc"
        )
    ]
)


lstm_callbacks = [

    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6
    )
]


lstm_history = lstm_model.fit(
    X_fit,
    Y_fit,

    validation_data=(
        X_internal_val,
        Y_internal_val
    ),

    epochs=30,
    batch_size=64,

    class_weight=class_weights,

    callbacks=lstm_callbacks,

    shuffle=False,

    verbose=1
)

Epoch 1/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 8s 36ms/step - accuracy: 0.8227 - auc: 0.6781 - loss: 0.6069 - precision: 0.2620 - recall: 0.1275 - val_accuracy: 0.6330 - val_auc: 0.5127 - val_loss: 0.6739 - val_precision: 0.6119 - val_recall: 0.9877 - learning_rate: 5.0000e-04
Epoch 2/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 4s 46ms/step - accuracy: 0.8091 - auc: 0.8099 - loss: 0.4347 - precision: 0.3588 - recall: 0.4142 - val_accuracy: 0.6486 - val_auc: 0.7100 - val_loss: 0.6241 - val_precision: 0.6276 - val_recall: 0.9545 - learning_rate: 5.0000e-04
Epoch 3/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.8260 - auc: 0.8577 - loss: 0.3860 - precision: 0.4195 - recall: 0.5429 - val_accuracy: 0.6556 - val_auc: 0.6753 - val_loss: 0.6225 - val_precision: 0.6330 - val_recall: 0.9533 - learning_rate: 5.0000e-04
Epoch 4/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.8368 - auc: 0.8789 - loss: 0.3571 - precision: 0.4512 - recall: 0.6176 - val_accuracy: 0.6796 - val_auc: 0.6967 - val_loss: 0.61

In [151]:
# Create internal train/validation split
# Final test set remains completely untouched

split_index = int(
    len(X_train_seq) * 0.80
)

X_fit = X_train_seq[
    :split_index
]

Y_fit = Y_train_seq[
    :split_index
]

X_internal_val = X_train_seq[
    split_index:
]

Y_internal_val = Y_train_seq[
    split_index:
]


print(
    "Model training:",
    X_fit.shape
)

print(
    "Internal validation:",
    X_internal_val.shape
)

print(
    "Final test:",
    X_test_seq.shape
)

print(
    "\nTraining labels:"
)

print(
    pd.Series(
        Y_fit
    ).value_counts()
)

print(
    "\nInternal validation labels:"
)

print(
    pd.Series(
        Y_internal_val
    ).value_counts()
)

Model training: (5667, 10, 57)
Internal validation: (1417, 10, 57)
Final test: (2125, 10, 57)

Training labels:
0    4851
1     816
Name: count, dtype: int64

Internal validation labels:
1    814
0    603
Name: count, dtype: int64


In [152]:
# LSTM Evaluation on Training Data

lstm_eval_prob = lstm_model.predict(
    X_train_seq,
    verbose=0
).ravel()

lstm_eval_pred = (
    lstm_eval_prob >= 0.5
).astype(int)

print("===== LSTM - Evaluation =====")

print(
    "Accuracy:",
    metrics.accuracy_score(
        Y_train_seq,
        lstm_eval_pred
    )
)

print(
    "Confusion Matrix:\n",
    metrics.confusion_matrix(
        Y_train_seq,
        lstm_eval_pred
    )
)

print(
    "Classification Report:\n",
    metrics.classification_report(
        Y_train_seq,
        lstm_eval_pred,
        digits=4,
        zero_division=0
    )
)

===== LSTM - Evaluation =====
Accuracy: 0.7567758328627894
Confusion Matrix:
 [[3980 1474]
 [ 249 1381]]
Classification Report:
               precision    recall  f1-score   support

           0     0.9411    0.7297    0.8221      5454
           1     0.4837    0.8472    0.6158      1630

    accuracy                         0.7568      7084
   macro avg     0.7124    0.7885    0.7189      7084
weighted avg     0.8359    0.7568    0.7746      7084



### LSTM Validation

In [153]:
# LSTM Validation on Test Data

lstm_val_prob = lstm_model.predict(
    X_test_seq,
    verbose=0
).ravel()

lstm_val_pred = (
    lstm_val_prob >= 0.5
).astype(int)

print("===== LSTM - Validation =====")

print(
    "Accuracy:",
    metrics.accuracy_score(
        Y_test_seq,
        lstm_val_pred
    )
)

print(
    "Confusion Matrix:\n",
    metrics.confusion_matrix(
        Y_test_seq,
        lstm_val_pred
    )
)

print(
    "Classification Report:\n",
    metrics.classification_report(
        Y_test_seq,
        lstm_val_pred,
        digits=4,
        zero_division=0
    )
)

===== LSTM - Validation =====
Accuracy: 0.936
Confusion Matrix:
 [[1783   93]
 [  43  206]]
Classification Report:
               precision    recall  f1-score   support

           0     0.9765    0.9504    0.9633      1876
           1     0.6890    0.8273    0.7518       249

    accuracy                         0.9360      2125
   macro avg     0.8327    0.8889    0.8575      2125
weighted avg     0.9428    0.9360    0.9385      2125



### Train and Evaluate GRU

In [154]:
gru_model = Sequential([
    Input(
        shape=(
            X_train_seq.shape[1],
            X_train_seq.shape[2]
        )
    ),

    GRU(
        128,
        return_sequences=True
    ),

    BatchNormalization(),

    Dropout(
        0.25
    ),

    GRU(
        64
    ),

    BatchNormalization(),

    Dropout(
        0.25
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(
        0.20
    ),

    Dense(
        32,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])


gru_model.compile(
    optimizer=Adam(
        learning_rate=0.0005
    ),

    loss="binary_crossentropy",

    metrics=[
        "accuracy",

        tf.keras.metrics.Precision(
            name="precision"
        ),

        tf.keras.metrics.Recall(
            name="recall"
        ),

        tf.keras.metrics.AUC(
            name="auc"
        )
    ]
)


gru_callbacks = [

    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6
    )
]


gru_history = gru_model.fit(
    X_fit,
    Y_fit,

    validation_data=(
        X_internal_val,
        Y_internal_val
    ),

    epochs=30,
    batch_size=64,

    class_weight=class_weights,

    callbacks=gru_callbacks,

    shuffle=False,

    verbose=1
)

Epoch 1/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 8s 33ms/step - accuracy: 0.7759 - auc: 0.6110 - loss: 0.6349 - precision: 0.2119 - recall: 0.2047 - val_accuracy: 0.6359 - val_auc: 0.6810 - val_loss: 0.6493 - val_precision: 0.6221 - val_recall: 0.9324 - learning_rate: 5.0000e-04
Epoch 2/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.8050 - auc: 0.7947 - loss: 0.4539 - precision: 0.3612 - recall: 0.4608 - val_accuracy: 0.6457 - val_auc: 0.6939 - val_loss: 0.6285 - val_precision: 0.6912 - val_recall: 0.6929 - learning_rate: 5.0000e-04
Epoch 3/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 6s 46ms/step - accuracy: 0.8244 - auc: 0.8502 - loss: 0.3968 - precision: 0.4180 - recall: 0.5588 - val_accuracy: 0.6493 - val_auc: 0.6978 - val_loss: 0.6248 - val_precision: 0.6974 - val_recall: 0.6880 - learning_rate: 5.0000e-04
Epoch 4/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.8244 - auc: 0.8626 - loss: 0.3789 - precision: 0.4234 - recall: 0.6066 - val_accuracy: 0.6443 - val_auc: 0.6904 - val_loss: 0.69

In [155]:
# GRU Evaluation on Training Data

gru_eval_prob = gru_model.predict(
    X_train_seq,
    verbose=0
).ravel()

gru_eval_pred = (
    gru_eval_prob >= THRESHOLD
).astype(int)

print("===== GRU - Evaluation =====")

print(
    "Accuracy:",
    metrics.accuracy_score(
        Y_train_seq,
        gru_eval_pred
    )
)

print(
    "Confusion Matrix:\n",
    metrics.confusion_matrix(
        Y_train_seq,
        gru_eval_pred
    )
)

print(
    "Classification Report:\n",
    metrics.classification_report(
        Y_train_seq,
        gru_eval_pred,
        digits=4,
        zero_division=0
    )
)

===== GRU - Evaluation =====
Accuracy: 0.7970073404856014
Confusion Matrix:
 [[4657  797]
 [ 641  989]]
Classification Report:
               precision    recall  f1-score   support

           0     0.8790    0.8539    0.8663      5454
           1     0.5538    0.6067    0.5790      1630

    accuracy                         0.7970      7084
   macro avg     0.7164    0.7303    0.7226      7084
weighted avg     0.8042    0.7970    0.8002      7084



### GRU Validation

In [156]:
# GRU Validation - Test Data

gru_val_prob = gru_model.predict(
    X_test_seq,
    verbose=0
).ravel()

gru_val_pred = (
    gru_val_prob >= THRESHOLD
).astype(int)

print("===== GRU - Validation =====")

print(
    "Accuracy:",
    metrics.accuracy_score(
        Y_test_seq,
        gru_val_pred
    )
)

print(
    "Confusion Matrix:\n",
    metrics.confusion_matrix(
        Y_test_seq,
        gru_val_pred
    )
)

print(
    "Classification Report:\n",
    metrics.classification_report(
        Y_test_seq,
        gru_val_pred,
        digits=4,
        zero_division=0
    )
)

===== GRU - Validation =====
Accuracy: 0.9270588235294117
Confusion Matrix:
 [[1841   35]
 [ 120  129]]
Classification Report:
               precision    recall  f1-score   support

           0     0.9388    0.9813    0.9596      1876
           1     0.7866    0.5181    0.6247       249

    accuracy                         0.9271      2125
   macro avg     0.8627    0.7497    0.7922      2125
weighted avg     0.9210    0.9271    0.9204      2125



### Results Presentation

In [157]:
# Recreate ML predictions with the correct datasets

# TRAIN / Evaluation predictions

lgr_eval_prob = lgr_model.predict_proba(
    X_train_temporal
)[:, 1]

lgr_eval_pred = (
    lgr_eval_prob >= LGR_THRESHOLD
).astype(int)


xgb_eval_prob = xgb_model.predict_proba(
    X_train_temporal
)[:, 1]

xgb_eval_pred = (
    xgb_eval_prob >= XGB_THRESHOLD
).astype(int)


# TEST / Validation predictions

lgr_val_prob = lgr_model.predict_proba(
    X_test_temporal
)[:, 1]

lgr_val_pred = (
    lgr_val_prob >= LGR_THRESHOLD
).astype(int)


xgb_val_prob = xgb_model.predict_proba(
    X_test_temporal
)[:, 1]

xgb_val_pred = (
    xgb_val_prob >= XGB_THRESHOLD
).astype(int)


print("TRAIN")
print("Y_train_seq:", len(Y_train_seq))
print("lgr_eval_pred:", len(lgr_eval_pred))
print("xgb_eval_pred:", len(xgb_eval_pred))

print("\nTEST")
print("Y_test_seq:", len(Y_test_seq))
print("lgr_val_pred:", len(lgr_val_pred))
print("xgb_val_pred:", len(xgb_val_pred))

TRAIN
Y_train_seq: 7084
lgr_eval_pred: 7084
xgb_eval_pred: 7084

TEST
Y_test_seq: 2125
lgr_val_pred: 2125
xgb_val_pred: 2125


In [158]:
# TRAIN RESULTS

train_results = pd.DataFrame([
    {
        "Model": "LSTM",
        "Accuracy": metrics.accuracy_score(
            Y_train_seq,
            lstm_eval_pred
        ),
        "Macro Precision": metrics.precision_score(
            Y_train_seq,
            lstm_eval_pred,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": metrics.recall_score(
            Y_train_seq,
            lstm_eval_pred,
            average="macro",
            zero_division=0
        ),
        "Macro F1": metrics.f1_score(
            Y_train_seq,
            lstm_eval_pred,
            average="macro",
            zero_division=0
        )
    },

    {
        "Model": "GRU",
        "Accuracy": metrics.accuracy_score(
            Y_train_seq,
            gru_eval_pred
        ),
        "Macro Precision": metrics.precision_score(
            Y_train_seq,
            gru_eval_pred,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": metrics.recall_score(
            Y_train_seq,
            gru_eval_pred,
            average="macro",
            zero_division=0
        ),
        "Macro F1": metrics.f1_score(
            Y_train_seq,
            gru_eval_pred,
            average="macro",
            zero_division=0
        )
    },

    {
        "Model": "Logistic Regression",
        "Accuracy": metrics.accuracy_score(
            Y_train_seq,
            lgr_eval_pred
        ),
        "Macro Precision": metrics.precision_score(
            Y_train_seq,
            lgr_eval_pred,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": metrics.recall_score(
            Y_train_seq,
            lgr_eval_pred,
            average="macro",
            zero_division=0
        ),
        "Macro F1": metrics.f1_score(
            Y_train_seq,
            lgr_eval_pred,
            average="macro",
            zero_division=0
        )
    },

    {
        "Model": "XGBoost",
        "Accuracy": metrics.accuracy_score(
            Y_train_seq,
            xgb_eval_pred
        ),
        "Macro Precision": metrics.precision_score(
            Y_train_seq,
            xgb_eval_pred,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": metrics.recall_score(
            Y_train_seq,
            xgb_eval_pred,
            average="macro",
            zero_division=0
        ),
        "Macro F1": metrics.f1_score(
            Y_train_seq,
            xgb_eval_pred,
            average="macro",
            zero_division=0
        )
    }
])

print("TRAIN RESULTS")
display(train_results.round(4))

print(" ")


# TEST RESULTS

test_results = pd.DataFrame([
    {
        "Model": "LSTM",
        "Accuracy": metrics.accuracy_score(
            Y_test_seq,
            lstm_val_pred
        ),
        "Macro Precision": metrics.precision_score(
            Y_test_seq,
            lstm_val_pred,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": metrics.recall_score(
            Y_test_seq,
            lstm_val_pred,
            average="macro",
            zero_division=0
        ),
        "Macro F1": metrics.f1_score(
            Y_test_seq,
            lstm_val_pred,
            average="macro",
            zero_division=0
        )
    },

    {
        "Model": "GRU",
        "Accuracy": metrics.accuracy_score(
            Y_test_seq,
            gru_val_pred
        ),
        "Macro Precision": metrics.precision_score(
            Y_test_seq,
            gru_val_pred,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": metrics.recall_score(
            Y_test_seq,
            gru_val_pred,
            average="macro",
            zero_division=0
        ),
        "Macro F1": metrics.f1_score(
            Y_test_seq,
            gru_val_pred,
            average="macro",
            zero_division=0
        )
    },

    {
        "Model": "Logistic Regression",
        "Accuracy": metrics.accuracy_score(
            Y_test_seq,
            lgr_val_pred
        ),
        "Macro Precision": metrics.precision_score(
            Y_test_seq,
            lgr_val_pred,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": metrics.recall_score(
            Y_test_seq,
            lgr_val_pred,
            average="macro",
            zero_division=0
        ),
        "Macro F1": metrics.f1_score(
            Y_test_seq,
            lgr_val_pred,
            average="macro",
            zero_division=0
        )
    },

    {
        "Model": "XGBoost",
        "Accuracy": metrics.accuracy_score(
            Y_test_seq,
            xgb_val_pred
        ),
        "Macro Precision": metrics.precision_score(
            Y_test_seq,
            xgb_val_pred,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": metrics.recall_score(
            Y_test_seq,
            xgb_val_pred,
            average="macro",
            zero_division=0
        ),
        "Macro F1": metrics.f1_score(
            Y_test_seq,
            xgb_val_pred,
            average="macro",
            zero_division=0
        )
    }
])

print("TEST RESULTS")
display(test_results.round(4))

TRAIN RESULTS


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
0,LSTM,0.7568,0.7124,0.7885,0.7189
1,GRU,0.7970,0.7164,0.7303,0.7226
2,Logistic Regression,0.7514,0.7287,0.8201,0.7239
3,XGBoost,0.8931,0.8547,0.8373,0.8455


 
TEST RESULTS


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
0,LSTM,0.9360,0.8327,0.8889,0.8575
1,GRU,0.9271,0.8627,0.7497,0.7922
2,Logistic Regression,0.9120,0.7832,0.9118,0.8282
3,XGBoost,0.9096,0.7988,0.7137,0.7464
